# Entraînement sur Colab — Diagnostic Mammographique par IA

Runtime recommandé : **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

Étapes : cloner le repo → installer → récupérer MVKL/ViKL → adapter les annotations →
pipeline de données → **smoke test** (1 epoch) → entraînement complet → checkpoint sur Drive.

> Le smoke test sert au **point 3** : premier run réel du code modèle pour débusquer les bugs
> (formes de tenseurs, tag `pretrained` d'open_clip, chemins d'images…). Colle-moi les erreurs
> éventuelles et on itère.

In [ ]:
!nvidia-smi -L
!python --version

In [ ]:
!git clone https://github.com/artichautttt/mammo.git
%cd mammo
!pip install -q -r requirements.txt

## 1. Récupérer le dataset MVKL / ViKL

Le dataset est public : lien SharePoint dans le README de https://github.com/wxwxwwxxx/ViKL
Télécharge l'archive, dépose-la sur ton Google Drive, puis :

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# adapte le nom de l'archive
!mkdir -p data/raw/MVKL
!unzip -q -o /content/drive/MyDrive/MVKL.zip -d data/raw/MVKL
!ls -R data/raw/MVKL | head -40

## 2. Adapter les annotations xlsx → mvkl_labeled_views.csv

On adapte **train ET validation** (et test si tu veux), puis on concatène en un
seul CSV portant la colonne `split`. `src/train_report_ready.py` utilise ce
`split` explicite dès qu'il contient `train` + `val` ; sinon il découpe
aléatoirement.

Ajuste les noms `--xlsx`, `--images-root`, et si besoin `--dcm-dir`,
`--birads-col`, `--manifestation-start` selon le `Readme.md` de l'archive.

In [ ]:
# --- train ---
!python -m scripts.adapt_vikl_annotations \
    --xlsx data/raw/MVKL/annotation/train_final_onehot.xlsx \
    --images-root data/raw/MVKL --split train \
    --out data/processed/mvkl_views_train.csv \
    --manifestations-out data/processed/manifestations.json

# --- validation ---
!python -m scripts.adapt_vikl_annotations \
    --xlsx data/raw/MVKL/annotation/val_final_onehot.xlsx \
    --images-root data/raw/MVKL --split val \
    --out data/processed/mvkl_views_val.csv \
    --manifestations-out data/processed/manifestations.json

# --- test (optionnel) ---
# !python -m scripts.adapt_vikl_annotations \
#     --xlsx data/raw/MVKL/annotation/test_final_onehot.xlsx \
#     --images-root data/raw/MVKL --split test \
#     --out data/processed/mvkl_views_test.csv \
#     --manifestations-out data/processed/manifestations.json

import glob
import pandas as pd

parts = [pd.read_csv(f) for f in sorted(glob.glob('data/processed/mvkl_views_*.csv'))]
views = pd.concat(parts, ignore_index=True)
views.to_csv('data/processed/mvkl_labeled_views.csv', index=False)
print(views.shape)
print(views['split'].value_counts().to_string())
views.head()

## 3. Pipeline de données (MVKL seul)

In [ ]:
!python -m scripts.build_combined_dual_view_dataset \
    --mvkl data/processed/mvkl_labeled_views.csv \
    --out  data/processed/combined_dual_view_cases.csv

!python -m scripts.filter_combined_dataset \
    --in  data/processed/combined_dual_view_cases.csv \
    --out data/processed/combined_dual_view_cases_filtered.csv

!python -m scripts.build_hybrid_augmented_weighted_dataset \
    --in  data/processed/combined_dual_view_cases_filtered.csv \
    --out data/processed/combined_hybrid_augmented_weighted.csv

## 4. Smoke test — 1 epoch sur un petit échantillon

But : vérifier que `DualViewCLIP`, le DataLoader, la perte multi-tâches et la boucle
tournent réellement de bout en bout. On restreint à 200 lignes et 1 epoch.

In [ ]:
import pandas as pd
d = pd.read_csv('data/processed/combined_hybrid_augmented_weighted.csv')
d.sample(min(200, len(d)), random_state=0).to_csv('data/processed/_smoke.csv', index=False)

!python -m src.train_report_ready \
    --dataset data/processed/_smoke.csv \
    --image-root . --epochs 1 --batch-size 4 \
    --out /content/_smoke_ckpt.pt

## 5. Entraînement complet

15–20 epochs max (au-delà : surapprentissage). Le meilleur checkpoint est retenu
dans la fenêtre epochs 12–15 et écrit dans `--out`.

In [ ]:
!mkdir -p /content/drive/MyDrive/mammo_ckpts
!python -m src.train_report_ready \
    --dataset data/processed/combined_hybrid_augmented_weighted.csv \
    --image-root . --epochs 18 --batch-size 32 \
    --out /content/drive/MyDrive/mammo_ckpts/best_two_view_model.pt

## 6. Index de similarité + test de prédiction

In [ ]:
!python -m src.similarity_search \
    --model /content/drive/MyDrive/mammo_ckpts/best_two_view_model.pt \
    --bank data/processed/combined_hybrid_augmented_weighted.csv \
    --image-root . --out /content/drive/MyDrive/mammo_ckpts/similarity_index

# exemple de prédiction (remplace par de vraies images)
!python -m src.two_view_model predict \
    --model /content/drive/MyDrive/mammo_ckpts/best_two_view_model.pt \
    --left  data/raw/MVKL/<...>/LMLO.a.png \
    --right data/raw/MVKL/<...>/LCC.a.png